# Lab 4 — Cloud IaC & Circuit Breakers

**Goal:** Understand the cloud deployment templates and implement the circuit breaker pattern.

**What's covered:**
1. How the Terraform (AWS), Cloud Run YAML (GCP), and Bicep (Azure) templates work
2. Circuit breaker: fail fast when the LLM API is down
3. Choosing the right cloud for your use case

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from circuit_breaker import CircuitBreaker, CircuitBreakerError, CircuitState
print('Ready ✓')

## 1. Inspect the cloud templates

In [ ]:
print('Cloud deployment options:\n')
files = {
    'AWS Lambda + API Gateway': 'deploy/aws_lambda.tf',
    'GCP Cloud Run':             'deploy/gcp_cloudrun.yaml',
    'Azure Container Apps':      'deploy/azure_container_apps.bicep',
}
for name, path in files.items():
    size = os.path.getsize(path) if os.path.exists(path) else 0
    print(f'  {name:30} {path}  ({size} bytes)')

print('\nKey differences:')
print('  AWS Lambda: pay-per-request, 15min timeout, cold starts ~100-300ms')
print('  GCP Cloud Run: scales to zero, persistent HTTP, better for long conversations')
print('  Azure Container Apps: Kubernetes-managed, DAPR support, best for microservices')

## 2. Circuit breaker — state machine demo

In [ ]:
from openai import OpenAI
client = OpenAI()

fail_counter = [0]

def unreliable_llm(prompt: str) -> str:
    fail_counter[0] += 1
    # Fail on calls 3, 4, 5 to simulate an outage
    if 3 <= fail_counter[0] <= 5:
        raise ConnectionError(f'LLM API unavailable (call {fail_counter[0]})')
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=30,
    )
    return resp.choices[0].message.content


cb = CircuitBreaker(
    fn=unreliable_llm,
    failure_threshold=2,
    recovery_timeout=2.0,   # short for demo
    fallback=lambda p, **kw: 'Service temporarily unavailable.',
)

import time

for i in range(10):
    try:
        result = cb(f'Question {i+1}: What is {i}+1?')
        print(f'[{i+1:2}] {cb.state.value:10} {result[:50]}')
    except Exception as e:
        print(f'[{i+1:2}] {cb.state.value:10} ERROR: {e}')
    
    if i == 6:  # wait for recovery timeout
        print('       Waiting for recovery timeout...')
        time.sleep(2.5)

## 3. Circuit breaker stats

In [ ]:
import json
stats = cb.stats()
print('Circuit breaker stats:')
print(json.dumps(stats, indent=2))

short_circuit_rate = stats['total_short_circuits'] / stats['total_calls']
print(f'\nShort circuit rate: {short_circuit_rate:.0%} of requests were failed fast (no API call made)')

## 4. Cloud comparison — choosing the right platform

In [ ]:
comparison = {
    'AWS Lambda': {
        'best_for': 'Bursty, stateless requests. Existing AWS infrastructure.',
        'free_tier': '1M requests/month, 400K GB-seconds',
        'max_timeout': '15 minutes',
        'cold_start': '100-300ms (container image: 1-3s)',
        'scaling': 'Near-instant, 1000 concurrent by default',
        'complexity': 'Medium (Terraform, SAM, or CDK needed)',
    },
    'GCP Cloud Run': {
        'best_for': 'HTTP-based agents, scale-to-zero, Google ecosystem.',
        'free_tier': '2M requests/month, 360K vCPU-seconds',
        'max_timeout': '60 minutes',
        'cold_start': '1-3s',
        'scaling': 'Scale to zero, up to 1000 instances',
        'complexity': 'Low (single YAML, gcloud CLI)',
    },
    'Azure Container Apps': {
        'best_for': 'Enterprise Microsoft shops, DAPR microservices, Kubernetes.',
        'free_tier': '180K vCPU-seconds, 360K GB-seconds/month',
        'max_timeout': 'No hard limit (configurable)',
        'cold_start': '1-5s',
        'scaling': 'KEDA-based, supports custom metrics',
        'complexity': 'Medium (Bicep/ARM, Azure CLI)',
    },
}

for platform, details in comparison.items():
    print(f'\n{platform}:')
    for k, v in details.items():
        print(f'  {k:15} {v}')

## Exercise
Combine all middleware into a production deployment test:  
1. Stack: `AuditLogger → PIIScrubber → BudgetMiddleware → CircuitBreaker → base_agent`
2. Send 10 requests, including: 2 with PII, 1 that exceeds budget, 1 during a simulated outage
3. Print the final audit log, budget report, and circuit breaker stats

This is the complete production middleware stack.

In [ ]:
# Your code here — the complete production stack
